# 2.14 深度学习训练基础 (Deep Learning Training Basics)

深度学习是大模型所有技术的基础，其核心训练机制支撑着上述所有学习范式。理解这些基础对于调试训练问题、优化训练效率至关重要。

本节涵盖：
- 反向传播与自动微分
- 优化器深入分析
- 学习率调度
- 正则化（权重衰减、Dropout变体）
- 归一化（LayerNorm vs RMSNorm）
- 梯度管理（裁剪、累积）
- 混合精度训练

## 1. 反向传播与自动微分

**目的**：高效计算神经网络中所有参数的梯度

**基本原理**：PyTorch使用动态计算图（Define-by-Run），每次前向传播时自动构建计算图，反向传播时沿计算图反向遍历，使用链式法则逐层计算偏导数。

**关键概念**：
- **计算图**：有向无环图（DAG），节点是张量操作，边是数据依赖
- **前向模式**：从输入到输出逐层计算，适合输入维度小于输出维度的场景
- **反向模式**：从输出到输入逐层计算梯度，适合输出维度小于输入维度的场景（神经网络通常如此）
- **雅可比向量积（JVP/VJP）**：自动微分的核心，反向传播计算VJP（向量-雅可比积）

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
X = torch.randn(4, 10)
y = torch.randint(0, 3, (4,))

loss_fn = nn.CrossEntropyLoss()
logits = model(X)
loss = loss_fn(logits, y)

print('=== Computational Graph & Backpropagation ===')
print(f'Loss: {loss.item():.4f}')
print(f'Loss grad_fn: {loss.grad_fn}')
print(f'Logits grad_fn: {logits.grad_fn}')
print(f'\nComputation graph (reverse mode):')
print(f'  loss <- CrossEntropy(logits, y)')
print(f'  logits <- Linear2(ReLU(Linear1(X)))')
print(f'  Backprop: dL/d(logits) -> dL/d(Linear2.w) -> dL/d(ReLU) -> dL/d(Linear1.w)')

loss.backward()

print(f'\nGradient shapes and norms:')
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f'  {name:20s}: shape={str(param.shape):15s} grad_norm={param.grad.norm():.4f}')

x = torch.randn(3, requires_grad=True)
y = x * 2
z = y.sum()
z.backward()
print(f'\nSimple example: x={x.data.tolist()}, dy/dx={x.grad.tolist()}')
print(f'Each element multiplied by 2, so gradient = 2 for each.')

print(f'\nKey: PyTorch builds the computation graph dynamically during forward pass.')
print(f'backward() traverses the graph in reverse, applying chain rule at each node.')

## 2. 优化器深入分析

**目的**：理解不同优化器的内部机制，选择适合场景的优化器

**SGD**：θ_{t+1} = θ_t - η·g_t，简单但需要精心调学习率

**SGD+Momentum**：v_t = β·v_{t-1} + g_t，θ_{t+1} = θ_t - η·v_t，累积动量加速收敛

**Adam**：维护一阶矩m和二阶矩v，自适应学习率。m_t = β₁·m_{t-1} + (1-β₁)·g_t，v_t = β₂·v_{t-1} + (1-β₂)·g_t²

**AdamW（关键）**：解耦权重衰减，是LLM训练的标准优化器。与Adam的区别：
- Adam：权重衰减通过L2正则化实现，梯度中包含衰减项，与自适应学习率耦合
- AdamW：权重衰减直接作用于参数，θ_t = (1-η·λ)·θ_t - η·m_t/(√v_t+ε)，与梯度更新解耦
- 实验表明AdamW在大多数任务上优于Adam，特别是大模型训练

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

def make_model():
    return nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))

model_sgd = make_model()
model_adam = make_model()
model_adamw = make_model()
model_adamw.load_state_dict(model_sgd.state_dict())
model_adam.load_state_dict(model_sgd.state_dict())

X = torch.randn(64, 10)
y = torch.randint(0, 3, (64,))
loss_fn = nn.CrossEntropyLoss()

opt_sgd = optim.SGD(model_sgd.parameters(), lr=0.1, momentum=0.9)
opt_adam = optim.Adam(model_adam.parameters(), lr=1e-3, weight_decay=0.01)
opt_adamw = optim.AdamW(model_adamw.parameters(), lr=1e-3, weight_decay=0.01)

print('=== Optimizer Comparison (with weight_decay=0.01) ===')
for epoch in range(30):
    for model, optimizer, name in [
        (model_sgd, opt_sgd, 'SGD+Mom'),
        (model_adam, opt_adam, 'Adam'),
        (model_adamw, opt_adamw, 'AdamW')
    ]:
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        losses = {}
        for model, name in [(model_sgd, 'SGD+Mom'), (model_adam, 'Adam'), (model_adamw, 'AdamW')]:
            with torch.no_grad():
                l = loss_fn(model(X), y).item()
            losses[name] = l
        print(f'Epoch {epoch+1}: ' + ', '.join(f'{k}={v:.4f}' for k, v in losses.items()))

adam_norm = sum(p.norm().item()**2 for p in model_adam.parameters())**0.5
adamw_norm = sum(p.norm().item()**2 for p in model_adamw.parameters())**0.5
print(f'\nParameter norms: Adam={adam_norm:.2f}, AdamW={adamw_norm:.2f}')
print(f'AdamW has smaller parameter norm due to decoupled weight decay.')
print(f'\nKey: AdamW is the standard optimizer for LLM training (GPT, LLaMA, etc.).')
print(f'Decoupled weight decay provides better regularization than L2 in Adam.')

## 3. 学习率调度

**目的**：在训练过程中动态调整学习率，平衡收敛速度和稳定性

**常用调度策略**：
- **线性预热+余弦衰减**：LLM训练最常用，先线性增大到峰值，再余弦衰减到接近0
- **WSD（Warmup-Stable-Decay）**：预热→稳定→衰减，DeepSeek系列使用
- **多项式衰减**：lr = lr₀ × (1 - t/T)^p，p控制衰减速度

**预热（Warmup）的必要性**：
- 训练初期，Adam的二阶矩v很小，自适应学习率可能过大
- 预热让v逐渐积累，避免初期参数更新过大导致训练不稳定
- 典型预热步数：总步数的1-5%，或1000-2000步

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

n_steps = 1000
warmup_steps = 100
peak_lr = 1e-3

def cosine_schedule(step, warmup, peak_lr, total_steps, min_lr=0.0):
    if step < warmup:
        return peak_lr * step / warmup
    progress = (step - warmup) / (total_steps - warmup)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

def wsd_schedule(step, warmup, stable_steps, peak_lr, total_steps, min_lr=0.0):
    if step < warmup:
        return peak_lr * step / warmup
    elif step < warmup + stable_steps:
        return peak_lr
    else:
        progress = (step - warmup - stable_steps) / (total_steps - warmup - stable_steps)
        return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

def polynomial_schedule(step, warmup, peak_lr, total_steps, power=2.0, min_lr=0.0):
    if step < warmup:
        return peak_lr * step / warmup
    progress = (step - warmup) / (total_steps - warmup)
    return min_lr + (peak_lr - min_lr) * (1 - progress) ** power

steps = list(range(0, n_steps, 50))
cos_lrs = [cosine_schedule(s, warmup_steps, peak_lr, n_steps) for s in steps]
wsd_lrs = [wsd_schedule(s, warmup_steps, 500, peak_lr, n_steps) for s in steps]
poly_lrs = [polynomial_schedule(s, warmup_steps, peak_lr, n_steps) for s in steps]

print('=== Learning Rate Schedules ===')
print(f'{"Step":>6} {"Cosine":>10} {"WSD":>10} {"Poly":>10}')
for s, c, w, p in zip(steps, cos_lrs, wsd_lrs, poly_lrs):
    print(f'{s:>6} {c:>10.6f} {w:>10.6f} {p:>10.6f}')

print(f'\nKey: Cosine decay is the default for LLM training.')
print(f'WSD keeps lr high longer (stable phase) then decays sharply.')
print(f'Warmup prevents training instability in early steps.')

## 4. 正则化：权重衰减与Dropout变体

**权重衰减**：防止参数过大导致过拟合
- **L2正则化**：损失函数中加入λ||θ||₂²，梯度中包含2λθ项，与优化器耦合
- **Decoupled Weight Decay**（AdamW）：直接衰减参数θ ← (1-ηλ)θ，与梯度更新解耦
- **工业实践**：LLM训练通常使用weight_decay=0.01-0.1，对bias和LayerNorm参数不施加

**Dropout变体**：
- **Standard Dropout**：随机置零神经元，训练时p概率丢弃，推理时不丢弃
- **DropPath/Stochastic Depth**：随机跳过整个残差块，训练深层网络的关键技术
- **Attention Dropout**：在注意力权重矩阵上施加dropout，防止注意力过拟合

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.1):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, device=x.device) < keep_prob
        return x * random_tensor / keep_prob

class ResidualBlockWithDropPath(nn.Module):
    def __init__(self, d=64, drop_path_prob=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )
        self.drop_path = DropPath(drop_path_prob)

    def forward(self, x):
        return x + self.drop_path(self.net(x))

x = torch.randn(4, 64)
block = ResidualBlockWithDropPath(drop_path_prob=0.2)

block.train()
out_train = block(x)
print('=== Dropout Variants ===')
print(f'DropPath (train mode): output has zeros at random samples')
print(f'  Input norm: {x.norm():.4f}')
print(f'  Output norm: {out_train.norm():.4f}')
print(f'  Zero rows: {(out_train == 0).all(dim=-1).sum().item()}/4')

block.eval()
out_eval = block(x)
print(f'\nDropPath (eval mode): no dropping')
print(f'  Output norm: {out_eval.norm():.4f}')

attn_weights = F.softmax(torch.randn(2, 4, 8, 8), dim=-1)
attn_dropout = nn.Dropout(p=0.1)
dropped = attn_dropout(attn_weights)
zero_count = (dropped == 0).sum().item()
total = dropped.numel()
print(f'\nAttention Dropout: {zero_count}/{total} weights zeroed ({zero_count/total:.1%})')

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if 'bias' in name or 'weight' in name and param.ndim < 2:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.01},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=1e-3)
print(f'\nWeight decay groups: {len(decay_params)} params with decay, {len(no_decay_params)} without')
print(f'\nKey: DropPath is essential for training deep transformers (ViT, Swin, LLaMA).')
print(f'Bias and LayerNorm params should NOT have weight decay (standard practice).')

## 5. 归一化：LayerNorm vs RMSNorm

**LayerNorm**：对每个token的隐层维度计算均值和方差，然后归一化。y = (x - μ) / √(σ² + ε) × γ + β

**RMSNorm**：LayerNorm的简化版，只计算均方根，不做均值减法。y = x / √(mean(x²) + ε) × γ

**关键区别**：
- RMSNorm省略了均值减法，计算更快（约减少10-15%计算量）
- RMSNorm不需要β偏置参数，参数量更少
- 实验表明RMSNorm与LayerNorm效果相当，但训练速度更快
- LLaMA、Mistral、DeepSeek等现代LLM均使用RMSNorm

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

x = torch.randn(4, 10)
layer_norm = nn.LayerNorm(10)
rms_norm = RMSNorm(10)

ln_out = layer_norm(x)
rn_out = rms_norm(x)

print('=== LayerNorm vs RMSNorm ===')
print(f'Input: mean={x.mean(dim=-1)[:4]}, std={x.std(dim=-1)[:4]}')
print(f'\nLayerNorm output: mean={ln_out.mean(dim=-1)[:4]}, std={ln_out.std(dim=-1)[:4]}')
print(f'RMSNorm output:   mean={rn_out.mean(dim=-1)[:4]}, std={rn_out.std(dim=-1)[:4]}')

ln_params = sum(p.numel() for p in layer_norm.parameters())
rn_params = sum(p.numel() for p in rms_norm.parameters())
print(f'\nLayerNorm params: {ln_params} (weight + bias)')
print(f'RMSNorm params:   {rn_params} (weight only, no bias)')

n_trials = 1000
x_large = torch.randn(32, 512, 1024)

import time
start = time.time()
for _ in range(n_trials):
    _ = nn.functional.layer_norm(x_large, [1024])
ln_time = time.time() - start

start = time.time()
for _ in range(n_trials):
    rms = torch.sqrt(x_large.pow(2).mean(dim=-1, keepdim=True) + 1e-8)
    _ = x_large / rms
rn_time = time.time() - start

print(f'\nSpeed: LayerNorm={ln_time:.3f}s, RMSNorm={rn_time:.3f}s (ratio={ln_time/rn_time:.2f}x)')
print(f'Note: Benchmark is relative; absolute timings depend on hardware. Pure PyTorch overhead affects results.')
print(f'\nKey: RMSNorm is faster (no mean subtraction) and used in LLaMA, Mistral, DeepSeek.')
print(f'For LLMs with many normalization layers, this speedup is significant.')

## 6. 梯度管理：裁剪与累积

**梯度裁剪**：防止梯度爆炸导致训练不稳定
- **按范数裁剪**：若||g|| > max_norm，则g ← g × max_norm / ||g||，是LLM训练的标配
- **按值裁剪**：将梯度值限制在[-clip_value, clip_value]，较少使用
- **典型设置**：max_norm=1.0，是GPT、LLaMA等模型的默认值

**梯度累积**：在显存有限时模拟大batch训练
- 将大batch分成多个小batch，分别计算梯度并累积
- 累积N步后执行一次优化器step，等效batch_size = micro_batch × N
- 是大模型训练的基本技术，几乎所有LLM训练都使用

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print('=== Gradient Clipping ===')
X = torch.randn(4, 10)
y = torch.randint(0, 3, (4,))
loss = F.cross_entropy(model(X), y)
loss.backward()

grad_norm = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None]).norm()
print(f'Gradient norm before clipping: {grad_norm:.4f}')

max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
grad_norm_after = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None]).norm()
print(f'Gradient norm after clipping (max_norm={max_norm}): {grad_norm_after:.4f}')
optimizer.step()
optimizer.zero_grad()

print(f'\n=== Gradient Accumulation ===')
model2 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)

accumulation_steps = 4
effective_batch = 64
micro_batch = effective_batch // accumulation_steps

X_full = torch.randn(effective_batch, 10)
y_full = torch.randint(0, 3, (effective_batch,))

optimizer2.zero_grad()
for micro_step in range(accumulation_steps):
    start = micro_step * micro_batch
    end = start + micro_batch
    X_micro = X_full[start:end]
    y_micro = y_full[start:end]
    loss = F.cross_entropy(model2(X_micro), y_micro) / accumulation_steps
    loss.backward()
    print(f'  Micro-step {micro_step+1}: loss={loss.item() * accumulation_steps:.4f}, '
          f'grad_norm={torch.cat([p.grad.flatten() for p in model2.parameters() if p.grad is not None]).norm():.4f}')

torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
optimizer2.step()

print(f'\nEffective batch_size: {effective_batch} = {micro_batch} micro_batch × {accumulation_steps} steps')
print(f'Loss is divided by {accumulation_steps} to match the scale of full-batch training.')
print(f'\nKey: Gradient accumulation enables large-batch training with limited GPU memory.')
print(f'GPT-4 training used gradient accumulation with micro_batch=1-4 on thousands of GPUs.')

## 7. 混合精度训练

**目的**：使用低精度数值格式加速训练，同时保持模型精度

**核心原理**：
- **FP32主权重**：维护FP32精度的主参数副本，用于参数更新
- **FP16/BF16前向+反向**：用低精度进行前向和反向传播，速度提升2-4倍
- **损失缩放（Loss Scaling）**：放大损失值防止小梯度在FP16下下溢为零

**FP16 vs BF16**：
- FP16：5位指数+10位尾数，范围小（6e-8到65504），需要损失缩放
- BF16：8位指数+7位尾数，范围与FP32相同（1e-38到3e38），不需要损失缩放
- **现代LLM训练首选BF16**：A100/H100原生支持，无需损失缩放

**FP8训练**：最新趋势，使用E4M3（前向）和E5M2（反向），在H100上实现2x BF16速度

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

print('=== Mixed Precision Training ===')

print('FP16 vs BF16 vs FP32 value ranges:')
print(f'  FP32: range [{torch.finfo(torch.float32).min:.2e}, {torch.finfo(torch.float32).max:.2e}], '
      f'min_positive={torch.finfo(torch.float32).tiny:.2e}')
print(f'  FP16: range [{torch.finfo(torch.float16).min:.2e}, {torch.finfo(torch.float16).max:.2e}], '
      f'min_positive={torch.finfo(torch.float16).tiny:.2e}')
if hasattr(torch, 'bfloat16'):
    print(f'  BF16: range [{torch.finfo(torch.bfloat16).min:.2e}, {torch.finfo(torch.bfloat16).max:.2e}], '
          f'min_positive={torch.finfo(torch.bfloat16).tiny:.2e}')

x_fp32 = torch.randn(4, 64)
x_fp16 = x_fp32.half()
x_bf16 = x_fp32.bfloat16()

print(f'\nPrecision comparison (same tensor in different formats):')
print(f'  FP32: {x_fp32[0, :5].tolist()}')
print(f'  FP16: {x_fp16[0, :5].tolist()}')
print(f'  BF16: {x_bf16[0, :5].tolist()}')

fp16_error = (x_fp32 - x_fp16.float()).abs().mean().item()
bf16_error = (x_fp32 - x_bf16.float()).abs().mean().item()
print(f'  FP16 avg error: {fp16_error:.8f}')
print(f'  BF16 avg error: {bf16_error:.8f}')
print(f'  BF16 has larger error per value but MUCH wider dynamic range (no overflow risk)')

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
model_fp32 = model
model_fp16 = model.half()

x = torch.randn(32, 64)

n_trials = 500
start = time.time()
for _ in range(n_trials):
    _ = model_fp32(x)
fp32_time = time.time() - start

start = time.time()
for _ in range(n_trials):
    _ = model_fp16(x.half())
fp16_time = time.time() - start

print(f'\nInference speed: FP32={fp32_time:.3f}s, FP16={fp16_time:.3f}s')
print(f'Speedup: {fp32_time/fp16_time:.2f}x')

print(f'\n=== Loss Scaling (for FP16) ===')
small_grad = torch.tensor([1e-6, 1e-7, 1e-8, 1e-9])
fp16_grad = small_grad.half()
print(f'FP32 gradients: {small_grad.tolist()}')
print(f'FP16 gradients: {fp16_grad.tolist()}')
print(f'Underflow: {(fp16_grad == 0).sum().item()}/{len(small_grad)} values lost!')

scale = 1024.0
scaled_grad = (small_grad * scale).half()
print(f'\nWith loss scaling (x{scale:.0f}):')
print(f'  Scaled FP16: {scaled_grad.tolist()}')
print(f'  Recovered:   {(scaled_grad.float() / scale).tolist()}')
print(f'  Underflow:   {((scaled_grad.float() / scale) == 0).sum().item()}/{len(small_grad)} values lost')

print(f'\nKey: BF16 is preferred for modern LLM training (no loss scaling needed).')
print(f'FP16 requires loss scaling to prevent gradient underflow.')
print(f'FP8 (E4M3/E5M2) is the next frontier for H100+ GPUs.')